# Préparation d'un jeu ProGEDO pour régression logistique — choix modal

**Source :** EMC² Toulouse 2023 (ProGEDO / lil-1750), fichiers standards `pers`, `men`, `depl`.
**Cible du projet principal :** chaque agent LLM (`llm-agents`) choisit, pour chaque déplacement,
un **mode** parmi `car / bike / walk / transit` (cf. `_primary_mode`, `simulation_controller.py`).

## Objectif
Construire un CSV **prêt pour une régression logistique multinomiale** dont :
- **une ligne = un déplacement** (unité de décision de l'agent) ;
- **la cible `y` = le mode principal** (`MODP` regroupé en `car/bike/walk/transit`) ;
- **les features `X` = uniquement les paramètres communs** avec le persona du projet principal
  (`traits_json` de `data/population/toulouse_population_*.json`), plus le **contexte de décision**
  (motif, distance, heure) qui entre dans le prompt de l'agent.

## Principe de « communalité »
On ne garde que ce qui est **strictement commun**, ou **rendu commun par transformation**
(recodage vers le même espace de valeurs que le projet principal).
Sont **exclus** : `income` et `employment_sector` (absents de ProGEDO — données sensibles / non captées).

| Trait projet principal | ProGEDO | Transformation |
|---|---|---|
| `age` | P4 | direct |
| `gender` | P2 | 1→Male, 2→Female |
| `household_size` | comptage personnes par (ZFP, ECH) | agrégat |
| `has_driving_license` | P7 | 1→True sinon False |
| `has_pt_subscription` | P12 | 4→False, 6→True |
| `number_of_cars` | M6 (ménage) | direct |
| `car_availability` | M6 + nb permis du foyer | none/some/all |
| `personal_bike` → `has_bike` | M21 (ménage) | M21>0 (M22 VAE non renseigné dans ce jeu) |
| `socioprofessional_class` | PCSC | recodage vers labels projet |
| `main_occupation` / `employed` / `studies` | P9 | recodage |
| `purpose` (contexte) | D5A | motif → home/work/education/shop/leisure/other |
| `distance_km` (contexte) | D12 | mètres → km |
| `departure_hour` (contexte) | D4 | HHMM → heure |


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 60)

# Racine du projet : ce notebook vit dans scripts/progedo_logit/
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "PROGEDO 2023").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PROGEDO_DIR = PROJECT_ROOT / "data" / "PROGEDO 2023" / "lil-1750-Donnees_CSV" / "fichiers_standards"
OUT_DIR = PROJECT_ROOT / "scripts" / "progedo_logit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("PROGEDO_DIR:", PROGEDO_DIR)
assert PROGEDO_DIR.exists(), "Dossier ProGEDO introuvable"


PROGEDO_DIR: /Users/yvesb/Documents/Projects/llm-agents-gama/data/PROGEDO 2023/lil-1750-Donnees_CSV/fichiers_standards


## 1. Import des trois fichiers standards

In [2]:
# Tout en str : les codes ProGEDO ont des zéros de tête ('01', '05'…) et des cellules
# vides significatives (' ' = non-enquêté). On recodera explicitement ensuite.
pers = pd.read_csv(PROGEDO_DIR / "Toulouse_2023_std_pers.csv", dtype=str)
men  = pd.read_csv(PROGEDO_DIR / "Toulouse_2023_std_men.csv",  dtype=str)
depl = pd.read_csv(PROGEDO_DIR / "Toulouse_2023_std_depl.csv", dtype=str)

# Nettoyage des espaces autour de tous les codes
for df in (pers, men, depl):
    for c in df.columns:
        df[c] = df[c].str.strip()

print("pers:", pers.shape, "| men:", men.shape, "| depl:", depl.shape)
depl.head(3)


pers: (20890, 45) | men: (10783, 77) | depl: (54585, 29)


,DP1,DMET,IDD3,IDD4,ZFD,ECH,PER,NDEP,GD1,STD,D2A,D2B,D3,GDO1,STDO,D4,D5A,D5B,D6,D7,GDD1,STDD,D8,D9,D10,D11,D12,MODP,TYPD
0,3,1,2023,31555,101101000,10001,1,1,31555,0101,01,,101101000,31555,0101,0800,25,,NaN,103101000,31555,0103,0820,20,1,1652,2478,33,1
1,3,1,2023,31555,101101000,10001,1,2,31555,0101,25,,103101000,31555,0103,1230,01,,NaN,101101000,31555,0101,1250,20,1,1652,2478,33,1
2,3,1,2023,31555,101101000,10001,1,3,31555,0101,01,,101101000,31555,0101,2030,51,,NaN,102103503,31555,0102,2042,12,1,939,1409,33,1


## 2. Dictionnaires de recodage (ProGEDO → espace de valeurs projet)

Les libellés proviennent de `lil-1750-Documentation/LABELS/`. On aligne sur les valeurs
observées dans `traits_json` du projet principal.

In [3]:
# --- P2 : sexe ---
GENDER = {"1": "Male", "2": "Female"}

# --- P9 : occupation principale → libellés projet ---
MAIN_OCCUPATION = {
    "1": "Travail à plein temps",
    "2": "Travail à temps partiel",
    "3": "Étudiant",                    # alternance / stage
    "4": "Étudiant",
    "5": "Scolaire (jusqu'au Bac)",
    "6": "Chômeur/recherche d'emploi",
    "7": "Retraité",
    "8": "Personne au foyer",
    "9": "Autre",
}
# employed / studies dérivés de P9
EMPLOYED_CODES = {"1", "2"}            # plein temps / temps partiel
STUDIES_CODES  = {"3", "4", "5"}      # alternance / étudiant / scolaire

# --- PCSC : catégorie socioprofessionnelle → labels projet principal ---
SOCIOPRO = {
    "01": "Farmer",
    "02": "Craftsperson or Shop Owner",
    "03": "Executive or Higher Intellectual Professional",
    "04": "Intermediate Professional",
    "05": "Employee",
    "06": "Manual Worker",
    "07": "Student",
    "08": "Other Inactive",
    "09": "Other Inactive",
}

# --- MODP : mode principal → cible {car, bike, walk, transit} ---
# Micro-mobilité active (roller/trottinette/fauteuil) rattachée à walk ;
# deux-roues motorisés, taxi/VTC, fourgon rattachés à car (motorisé privé) ;
# avion / fluvial / engins agricoles = hors champ urbain → écartés (None).
MODE_GROUP = {
    "01": "walk", "93": "walk", "94": "walk", "96": "walk", "97": "walk",
    "10": "bike", "11": "bike", "12": "bike", "17": "bike", "18": "bike",
    "21": "car", "22": "car", "61": "car", "62": "car", "81": "car", "82": "car",
    "13": "car", "14": "car", "15": "car", "16": "car", "19": "car", "20": "car",
    "31": "transit", "32": "transit", "33": "transit", "34": "transit",
    "37": "transit", "38": "transit", "39": "transit",
    "41": "transit", "42": "transit", "43": "transit",
    "51": "transit", "52": "transit", "53": "transit", "54": "transit",
    "71": "transit",
    # 91 fluvial, 92 avion, 95 engins agricoles → None (écartés)
}

# --- D5A : motif destination → purpose projet {home, work, education, shop, leisure, other} ---
def purpose_from_d5a(code: str) -> str:
    if code in ("01", "02"):                       return "home"
    if code in ("11", "12", "13", "14", "81"):     return "work"
    if code in ("21", "22", "23", "24", "25", "26", "27", "28", "29", "96", "97"):
        return "education"
    if code in ("30", "31", "32", "33", "34", "35", "82", "98"):
        return "shop"
    if code in ("51", "52", "53", "54"):           return "leisure"
    return "other"


## 3. Table ménage (`men`) — équipement du foyer

Clé ménage réelle = **(ZFM, ECH)** (`ECH` seul n'est pas unique).
On dérive `number_of_cars`, `has_bike`.

In [4]:
men_feat = pd.DataFrame({
    "ZF": men["ZFM"],
    "ECH": men["ECH"],
    "number_of_cars": pd.to_numeric(men["M6"], errors="coerce"),
    "n_bikes": pd.to_numeric(men["M21"], errors="coerce"),
})
men_feat["has_bike"] = men_feat["n_bikes"].fillna(0) > 0
# M22 (vélos électriques) non renseigné dans ce jeu → impossible de distinguer VAE.
print("M22 non-null:", men["M22"].notna().sum(), "→ personal_bike réduit à has_bike")
men_feat = men_feat.drop_duplicates(["ZF", "ECH"])
print(men_feat.shape)
men_feat.head(3)


M22 non-null: 0 → personal_bike réduit à has_bike
(10783, 5)


,ZF,ECH,number_of_cars,n_bikes,has_bike
0,101101000,10001,0,0,False
1,101101000,10018,0,0,False
2,101101000,10019,1,0,False


## 4. Table personne (`pers`) — démographie & équipement individuel

Clé personne = **(ZFP, ECH, PER)**. `household_size` = nb de personnes par (ZFP, ECH).

In [5]:
# Taille du ménage = comptage des personnes par (ZFP, ECH)
hh_size = pers.groupby(["ZFP", "ECH"]).size().rename("household_size").reset_index()

# Nb de titulaires du permis par foyer → utile pour car_availability
lic_per_hh = (
    pers.assign(_lic=(pers["P7"] == "1"))
        .groupby(["ZFP", "ECH"])["_lic"].sum()
        .rename("n_licensed").reset_index()
)

pers_feat = pd.DataFrame({
    "ZF": pers["ZFP"],
    "ECH": pers["ECH"],
    "PER": pers["PER"],
    "PENQ": pers["PENQ"],
    "age": pd.to_numeric(pers["P4"], errors="coerce"),
    "gender": pers["P2"].map(GENDER),
    "has_driving_license": pers["P7"] == "1",
    "has_pt_subscription": pers["P12"].map({"4": False, "6": True}),
    "socioprofessional_class": pers["PCSC"].map(SOCIOPRO),
    "main_occupation": pers["P9"].map(MAIN_OCCUPATION),
    "employed": pers["P9"].isin(EMPLOYED_CODES),
    "studies": pers["P9"].isin(STUDIES_CODES),
})
pers_feat = (
    pers_feat
    .merge(hh_size,     left_on=["ZF", "ECH"], right_on=["ZFP", "ECH"], how="left").drop(columns="ZFP")
    .merge(lic_per_hh,  left_on=["ZF", "ECH"], right_on=["ZFP", "ECH"], how="left").drop(columns="ZFP")
    .merge(men_feat[["ZF", "ECH", "number_of_cars", "has_bike"]], on=["ZF", "ECH"], how="left")
)

# car_availability (none/some/all), sémantique projet : offre voitures vs conducteurs du foyer
def car_availability(row):
    cars = row["number_of_cars"]
    if pd.isna(cars):        return np.nan
    if cars == 0:            return "none"
    lic = row["n_licensed"]
    if pd.isna(lic) or lic == 0:  return "all"      # voiture dispo, pas de contrainte de conducteur
    return "all" if cars >= lic else "some"

pers_feat["car_availability"] = pers_feat.apply(car_availability, axis=1)
print(pers_feat.shape)
pers_feat.head(3)


(20890, 17)


,ZF,ECH,PER,PENQ,age,gender,has_driving_license,has_pt_subscription,socioprofessional_class,main_occupation,employed,studies,household_size,n_licensed,number_of_cars,has_bike,car_availability
0,101101000,10001,1,1,18,Male,False,True,Student,Étudiant,False,True,1,0,0,False,none
1,101101000,10018,1,1,23,Female,False,True,Student,Étudiant,False,True,2,0,0,False,none
2,101101000,10018,2,1,22,Male,False,True,Student,Étudiant,False,True,2,0,0,False,none


## 5. Merge déplacements + personne + ménage

`depl` (ZFD, ECH, PER) → `pers_feat` (ZF, ECH, PER). Le ménage est déjà porté par `pers_feat`.

In [6]:
dep = pd.DataFrame({
    "ZF": depl["ZFD"],
    "ECH": depl["ECH"],
    "PER": depl["PER"],
    "NDEP": depl["NDEP"],
    "MODP": depl["MODP"],
    "D5A": depl["D5A"],
    "D12_m": pd.to_numeric(depl["D12"], errors="coerce"),
    "D4": depl["D4"],
})

# Cible et features de contexte
dep["mode"] = dep["MODP"].map(MODE_GROUP)                       # cible
dep["purpose"] = dep["D5A"].map(purpose_from_d5a)              # contexte
dep["distance_km"] = dep["D12_m"] / 1000.0                     # contexte
dep["departure_hour"] = (pd.to_numeric(dep["D4"], errors="coerce") // 100) % 24  # contexte

df = dep.merge(pers_feat, on=["ZF", "ECH", "PER"], how="left")
print("Après merge:", df.shape)
print("Merge personne réussi:", df["age"].notna().mean().round(3))
df.head(3)


Après merge: (54585, 26)
Merge personne réussi: 1.0


,ZF,ECH,PER,NDEP,MODP,D5A,D12_m,D4,mode,purpose,distance_km,departure_hour,PENQ,age,gender,has_driving_license,has_pt_subscription,socioprofessional_class,main_occupation,employed,studies,household_size,n_licensed,number_of_cars,has_bike,car_availability
0,101101000,10001,1,1,33,25,2478,0800,transit,education,2.478,8,1,18,Male,False,True,Student,Étudiant,False,True,1,0,0,False,none
1,101101000,10001,1,2,33,01,2478,1230,transit,home,2.478,12,1,18,Male,False,True,Student,Étudiant,False,True,1,0,0,False,none
2,101101000,10001,1,3,33,51,1409,2030,transit,leisure,1.409,20,1,18,Male,False,True,Student,Étudiant,False,True,1,0,0,False,none


## 6. Nettoyage : lignes & colonnes inutiles

In [7]:
FEATURES = [
    # persona statique (strictement commun / rendu commun)
    "age", "gender", "household_size",
    "has_driving_license", "has_pt_subscription",
    "number_of_cars", "car_availability", "has_bike",
    "socioprofessional_class", "main_occupation", "employed", "studies",
    # contexte de décision (entre dans le prompt de l'agent)
    "purpose", "distance_km", "departure_hour",
]
TARGET = "mode"
KEYS = ["ZF", "ECH", "PER", "NDEP"]   # traçabilité, hors modèle

n0 = len(df)
# 1) cible exploitable (modes hors champ écartés)
df = df[df["mode"].notna()]
# 2) personne effectivement enquêtée (features individuelles renseignées)
df = df[df["PENQ"] == "1"]
# 3) features critiques non manquantes
df = df.dropna(subset=["age", "gender", "has_pt_subscription",
                       "socioprofessional_class", "main_occupation",
                       "car_availability", "number_of_cars", "distance_km"])
print(f"Lignes : {n0} → {len(df)}  (écartées : {n0 - len(df)})")

clean = df[KEYS + FEATURES + [TARGET]].reset_index(drop=True)

# Types propres
for c in ["has_driving_license", "has_pt_subscription", "has_bike", "employed", "studies"]:
    clean[c] = clean[c].astype(bool)
clean["age"] = clean["age"].astype(int)
clean["household_size"] = clean["household_size"].astype("Int64")
clean["number_of_cars"] = clean["number_of_cars"].astype("Int64")
clean["departure_hour"] = clean["departure_hour"].astype("Int64")

clean.head()


Lignes : 54585 → 54559  (écartées : 26)


,ZF,ECH,PER,NDEP,age,gender,household_size,has_driving_license,has_pt_subscription,number_of_cars,car_availability,has_bike,socioprofessional_class,main_occupation,employed,studies,purpose,distance_km,departure_hour,mode
0,101101000,10001,1,1,18,Male,1,False,True,0,none,False,Student,Étudiant,False,True,education,2.478,8,transit
1,101101000,10001,1,2,18,Male,1,False,True,0,none,False,Student,Étudiant,False,True,home,2.478,12,transit
2,101101000,10001,1,3,18,Male,1,False,True,0,none,False,Student,Étudiant,False,True,leisure,1.409,20,transit
3,101101000,10001,1,4,18,Male,1,False,True,0,none,False,Student,Étudiant,False,True,home,1.409,23,transit
4,101101000,10018,2,1,22,Male,2,False,True,0,none,False,Student,Étudiant,False,True,education,6.605,13,transit


In [8]:
# Contrôles de distribution
print("Cible (mode) :")
print(clean["mode"].value_counts(normalize=True).round(3))
print("\npurpose :")
print(clean["purpose"].value_counts(normalize=True).round(3))
print("\ncar_availability :")
print(clean["car_availability"].value_counts(normalize=True).round(3))
print("\nValeurs manquantes par colonne :")
print(clean.isna().sum()[lambda s: s > 0])


Cible (mode) :
mode
car        0.565
walk       0.278
transit    0.118
bike       0.039
Name: proportion, dtype: float64

purpose :
purpose
home         0.394
shop         0.142
leisure      0.138
work         0.134
other        0.125
education    0.068
Name: proportion, dtype: float64

car_availability :
car_availability
all     0.728
some    0.137
none    0.135
Name: proportion, dtype: float64

Valeurs manquantes par colonne :
Series([], dtype: int64)


## 7. Séparation features / cible et export CSV

In [9]:
X = clean[FEATURES].copy()
y = clean[TARGET].copy()

# CSV combiné (features + cible + clés de traçabilité) — entrée principale
csv_all = OUT_DIR / "progedo_mode_choice.csv"
clean.to_csv(csv_all, index=False)

# Variantes séparées si besoin d'un pipeline X / y distinct
X.to_csv(OUT_DIR / "progedo_mode_choice_X.csv", index=False)
y.to_frame().to_csv(OUT_DIR / "progedo_mode_choice_y.csv", index=False)

print("Écrits :")
print(" -", csv_all, f"({len(clean)} lignes, {clean.shape[1]} colonnes)")
print(" -", OUT_DIR / "progedo_mode_choice_X.csv", X.shape)
print(" -", OUT_DIR / "progedo_mode_choice_y.csv", y.shape)
X.dtypes


Écrits :
 - /Users/yvesb/Documents/Projects/llm-agents-gama/scripts/progedo_logit/progedo_mode_choice.csv (54559 lignes, 20 colonnes)
 - /Users/yvesb/Documents/Projects/llm-agents-gama/scripts/progedo_logit/progedo_mode_choice_X.csv (54559, 15)
 - /Users/yvesb/Documents/Projects/llm-agents-gama/scripts/progedo_logit/progedo_mode_choice_y.csv (54559,)


age                          int64
gender                         str
household_size               Int64
has_driving_license           bool
has_pt_subscription           bool
number_of_cars               Int64
car_availability               str
has_bike                      bool
socioprofessional_class        str
main_occupation                str
employed                      bool
studies                       bool
purpose                        str
distance_km                float64
departure_hour               Int64
dtype: object

## 8. Contrôle : régression logistique multinomiale (smoke test)

Vérifie que le CSV s'ajuste tel quel dans un pipeline scikit-learn
(one-hot des catégorielles + standardisation des numériques). Non destiné à l'analyse finale.

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

num_cols = ["age", "household_size", "number_of_cars", "distance_km", "departure_hour"]
cat_cols = ["gender", "car_availability", "socioprofessional_class", "main_occupation", "purpose"]
bool_cols = ["has_driving_license", "has_pt_subscription", "has_bike", "employed", "studies"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("bool", "passthrough", bool_cols),
])
clf = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced")),
])

Xtr, Xte, ytr, yte = train_test_split(X.astype({c: float for c in num_cols}),
                                      y, test_size=0.25, random_state=0, stratify=y)
clf.fit(Xtr, ytr)
print("Accuracy test :", round(clf.score(Xte, yte), 3))
print(classification_report(yte, clf.predict(Xte)))


Accuracy test : 0.686
              precision    recall  f1-score   support

        bike       0.14      0.61      0.23       531
         car       0.93      0.62      0.74      7704
     transit       0.57      0.72      0.63      1608
        walk       0.75      0.82      0.78      3797

    accuracy                           0.69     13640
   macro avg       0.60      0.69      0.60     13640
weighted avg       0.80      0.69      0.72     13640

